<a href="https://colab.research.google.com/github/NicheMobility/analisis-vibracions/blob/dev/Analasi_vibracions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚲⚙️ Anàlisi FFT de Vibracions — IMU al Motor de la Bicicleta

**Sensor:** ST ASM330LHHXG1 (acc ±mg, gyro ±mdps) · **Registre:** ST MEMS Studio · **Fs ≈ 440 Hz**

**Objectiu:** caracteritzar les vibracions mecàniques que **rep el motor** durant la conducció per definir un **perfil d'assaig d'envelliment ràpid** en banc de vibració.

| # | Secció | Resultat |
|---|--------|----------|
| 1–3 | Configuració | Llibreries, Drive, paràmetres |
| 4 | Funcions | FFT, PSD, 1/3 octava, g-RMS, FDS, pics |
| 5–6 | Dades | Càrrega, calibració, **correcció eix Z invertit** |
| 7 | FFT | Espectre d'amplitud (g pic) |
| 8 | PSD Welch | Densitat espectral (g²/Hz) |
| 9 | 1/3 d'octava | g-RMS per bandes ISO 266 |
| 10 | Espectrograma | Mapa temps-freqüència |
| 11 | Estadístiques | g-RMS, curtosi, factor de cresta |
| 12 | FDS | Espectre de dany per fatiga |
| 13 | Resum | Taula per configurar el banc de vibració |
| 14 | Export | CSV de resultats |

> ▶️ **Ús:** només cal canviar `CSV_FILENAME` (secció 3) i executar totes les cel·les (*Entorn d'execució → Executa-ho tot*).

## 📦 1 - Importació de llibreries

In [ ]:
"""
ANÀLISI FFT DE VIBRACIONS - IMU ASM330LHHXG1 AL MOTOR DE LA BICICLETA
Caracterització per a assaig d'envelliment ràpid en banc de vibració
"""

import os
import numpy as np
import pandas as pd

from scipy import signal, stats
from scipy.fft import rfft, rfftfreq
from scipy.special import gamma
from scipy.integrate import trapezoid

import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Paleta de colors (mateixa que el notebook del filtre complementari)
COLOR_X = '#2E86AB'   # blau
COLOR_Y = '#A23B72'   # magenta
COLOR_Z = '#F18F01'   # taronja
COLOR_T = '#3B3B3B'   # suma vectorial
COLORS = {'x': COLOR_X, 'y': COLOR_Y, 'z': COLOR_Z, 'total': COLOR_T}
PLOTLY_TEMPLATE = 'plotly_white'

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 30)

print("✓ Llibreries importades")

## 💾 2 - Google Drive

In [ ]:
# Muntar Google Drive (si s'executa a Colab; en local es fa servir el directori actual)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    # ⬇️ EDITA AQUESTA RUTA AMB EL TEU DIRECTORI ⬇️
    WORKING_DIR = '/content/drive/MyDrive/IMU/'
    IN_COLAB = True
except ImportError:
    WORKING_DIR = os.getcwd()
    IN_COLAB = False
    print("ℹ️ No s'ha detectat Google Colab → s'usa el directori local")

os.makedirs(WORKING_DIR, exist_ok=True)
os.chdir(WORKING_DIR)

print(f"✓ Google Drive muntat" if IN_COLAB else "✓ Mode local")
print(f"✓ Directori: {os.getcwd()}")

## ⚙️ 3 - Paràmetres

> ⚠️ **Límit de Nyquist:** amb Fs ≈ 440 Hz, la freqüència màxima representable és ≈ 220 Hz. L'anàlisi es limita a **200 Hz** per evitar la zona de transició del filtre anti-aliasing intern del sensor. Vibracions reals per sobre de 220 Hz (p. ex. engranatges ràpids del motor) **no es poden veure** i poden aparèixer replegades (*aliasing*) si el filtre intern no les atenua prou.

In [ ]:
# ⬇️ ESPECIFICA EL NOM DEL TEU FITXER ⬇️
CSV_FILENAME = 'Vib-01-perfil-sorra.csv'

# Descripció de l'assaig (només informativa, surt a la taula resum)
SENSOR_POSITION = 'Motor de la bicicleta (carcassa)'

# ---------------- Calibració ----------------
CALIBRATION_TIME = 5.0      # s · període estàtic inicial (bias gyro) que després es descarta

# ---------------- Conversions ----------------
MG_TO_G = 1.0 / 1000.0      # mg → g
G_TO_MS2 = 9.80665          # g → m/s²
MDPS_TO_DPS = 1.0 / 1000.0  # mdps → dps

# ---------------- FFT / PSD ----------------
NPERSEG = 2048              # mostres per segment de Welch (Δf ≈ Fs/NPERSEG ≈ 0.21 Hz)
WELCH_OVERLAP = 0.75        # solapament entre segments (75 %)
WINDOW = 'hann'             # finestra
F_MIN = 1.0                 # Hz · freqüència mínima d'anàlisi
F_MAX = 200.0               # Hz · freqüència màxima d'anàlisi (< Nyquist)
N_PEAKS = 10                # pics a detectar per canal

# Bandes de g-RMS per a les estadístiques
GRMS_BANDS = [(1.0, 10.0), (10.0, 50.0), (50.0, 200.0)]

# ---------------- Espectrograma ----------------
NPERSEG_SPECTROGRAM = NPERSEG   # redueix-lo (p. ex. 512) per a més resolució temporal

# ---------------- Fatigue Damage Spectrum ----------------
FDS_B = 6.0                 # exponent de Basquin (alumini ≈ 6; acers 8–12)
FDS_DAMPING = 0.05          # ζ · esmorteïment (Q = 1/(2ζ) = 10)
FDS_N_POINTS = 500          # freqüències naturals (escala log)

# ---------------- Envelliment accelerat ----------------
# Equivalència de dany (Basquin/Miner): T_assaig · G_assaig^b = T_real · G_real^b
HORES_US_REAL = 1000.0      # h · vida d'ús real que es vol reproduir
HORES_ASSAIG = 20.0         # h · durada disponible de l'assaig al banc

# ---------------- Bandes 1/3 d'octava (ISO 266, nominals) ----------------
OCTAVE_CENTERS = np.array([1.0, 1.25, 1.6, 2.0, 2.5, 3.15, 4.0, 5.0, 6.3, 8.0,
                           10.0, 12.5, 16.0, 20.0, 25.0, 31.5, 40.0, 50.0, 63.0,
                           80.0, 100.0, 125.0, 160.0, 200.0])

print("=" * 60)
print("PARÀMETRES D'ANÀLISI")
print("=" * 60)
print(f"Fitxer:             {CSV_FILENAME}")
print(f"Calibració:         {CALIBRATION_TIME} s")
print(f"Welch:              nperseg={NPERSEG}, solapament={WELCH_OVERLAP*100:.0f} %, finestra='{WINDOW}'")
print(f"Rang freqüencial:   {F_MIN} – {F_MAX} Hz")
print(f"FDS:                b={FDS_B}, ζ={FDS_DAMPING} (Q={1/(2*FDS_DAMPING):.0f})")
print(f"Bandes 1/3 octava:  {len(OCTAVE_CENTERS)} ({OCTAVE_CENTERS[0]} – {OCTAVE_CENTERS[-1]} Hz)")
print("=" * 60)

## 🧮 4 - Funcions d'anàlisi

- **FFT unilateral** amb finestra de Hann i correcció d'amplitud (coherent gain) → amplitud de pic en unitats del senyal.
- **PSD de Welch** (`scaling='density'`, `detrend='linear'`) → g²/Hz.
- **g-RMS** = √∫PSD·df (integració trapezoidal sobre una malla fina interpolada, robusta a bandes estretes).
- **FDS** (Lalanne, mètode de banda estreta): per a cada freqüència natural *fₙ* d'un oscil·lador d'1 GDL:
  - |H(f)|² = 1 / [ (1 − (f/fₙ)²)² + (2ζ f/fₙ)² ]  (pseudo-acceleració)
  - σ = √∫ |H(f)|² · PSD(f) df
  - D(fₙ) = T · fₙ · (√2 · σ)ᵇ · Γ(1 + b/2)

In [ ]:
def _band_integral(freqs, psd, f_low, f_high, n_grid=256):
    """Integra la PSD entre f_low i f_high interpolant sobre una malla fina.
    Evita errors quan la banda conté molt pocs bins (bandes baixes de 1/3 d'octava)."""
    f_low = max(f_low, freqs[0])
    f_high = min(f_high, freqs[-1])
    if f_high <= f_low:
        return 0.0
    f_grid = np.linspace(f_low, f_high, n_grid)
    return float(trapezoid(np.interp(f_grid, freqs, psd), f_grid))


def compute_fft_onesided(sig, fs):
    """FFT unilateral amb finestra de Hann i correcció d'amplitud.

    Args:
        sig: senyal (sense DC) en unitats físiques (g o dps)
        fs:  freqüència de mostreig [Hz]
    Returns:
        freqs [Hz], amplitud de pic [unitats del senyal]
    """
    sig = np.asarray(sig, dtype=float)
    n = len(sig)
    w = signal.get_window('hann', n)
    spec = rfft((sig - np.mean(sig)) * w)
    amp = np.abs(spec) / np.sum(w)          # correcció coherent gain de la finestra
    amp[1:] *= 2.0                          # unilateral: duplica tot excepte DC...
    if n % 2 == 0:
        amp[-1] /= 2.0                      # ...i Nyquist (si n és parell)
    freqs = rfftfreq(n, d=1.0 / fs)
    return freqs, amp


def compute_psd_welch(sig, fs, nperseg=NPERSEG, overlap=WELCH_OVERLAP, window=WINDOW):
    """PSD de Welch (densitat) → unitats²/Hz."""
    nperseg = int(min(nperseg, len(sig)))
    noverlap = int(nperseg * overlap)
    freqs, psd = signal.welch(sig, fs=fs, window=window, nperseg=nperseg,
                              noverlap=noverlap, detrend='linear',
                              scaling='density', return_onesided=True)
    return freqs, psd


def octave_band_edges(centers, fraction=3):
    """Límits de banda de fracció d'octava: f_low = fc/2^(1/2n), f_high = fc·2^(1/2n)."""
    factor = 2.0 ** (1.0 / (2 * fraction))     # 1/3 octava → 2^(1/6)
    centers = np.asarray(centers, dtype=float)
    return centers / factor, centers * factor


def compute_octave_band_rms(freqs, psd, centers, fraction=3):
    """RMS per banda de fracció d'octava a partir de la PSD.
    Returns: DataFrame amb fc, f_low, f_high, rms, psd_mitjana (unitats²/Hz)."""
    f_lo, f_hi = octave_band_edges(centers, fraction)
    nyq = freqs[-1]
    rows = []
    for fc, fl, fh in zip(centers, f_lo, f_hi):
        fh_eff = min(fh, nyq)
        ms = _band_integral(freqs, psd, fl, fh_eff)
        bw = max(fh_eff - fl, 1e-12)
        rows.append({'fc_Hz': fc, 'f_low_Hz': fl, 'f_high_Hz': fh,
                     'rms': np.sqrt(ms), 'psd_mitjana': ms / bw,
                     'truncada_nyquist': fh > nyq})
    return pd.DataFrame(rows)


def compute_grms(freqs, psd, f_low=F_MIN, f_high=F_MAX):
    """g-RMS (o RMS en les unitats del senyal) dins una banda: √∫PSD df."""
    return np.sqrt(_band_integral(freqs, psd, f_low, f_high, n_grid=4096))


def compute_fds(psd_values, freqs, natural_freqs, damping=FDS_DAMPING,
                duration=1.0, b=FDS_B, f_low=F_MIN, f_high=F_MAX):
    """Fatigue Damage Spectrum (Lalanne, aproximació de banda estreta).

    Per a cada fn: |H|² = 1/((1-r²)² + (2ζr)²), r = f/fn  (pseudo-acceleració ωn²·z)
                   σ = √∫|H|²·PSD df
                   D = T · fn · (√2·σ)^b · Γ(1+b/2)
    La constant del material (K^b/C) s'omet → el FDS és RELATIU (s'usa per comparar).

    La PSD s'interpola en una malla logarítmica fina perquè els pics de ressonància
    (amplada ≈ 2ζ·fn, molt estreta a baixa freqüència) quedin ben resolts.
    Returns: damage (absolut, unitats arbitràries), damage normalitzat al màxim.
    """
    f_fine = np.geomspace(max(f_low, freqs[1]), min(f_high, freqs[-1]), 20000)
    psd_fine = np.interp(f_fine, freqs, psd_values)
    damage = np.zeros(len(natural_freqs))
    for i, fn in enumerate(natural_freqs):
        r = f_fine / fn
        h2 = 1.0 / ((1.0 - r**2)**2 + (2.0 * damping * r)**2)
        sigma = np.sqrt(trapezoid(h2 * psd_fine, f_fine))
        damage[i] = duration * fn * (np.sqrt(2.0) * sigma)**b * gamma(1.0 + b / 2.0)
    norm = damage / np.max(damage) if np.max(damage) > 0 else damage
    return damage, norm


def find_spectral_peaks(freqs, spectrum, n_peaks=N_PEAKS, f_low=F_MIN, f_high=F_MAX,
                        min_distance_hz=1.0):
    """Detecta els n pics més prominents dins [f_low, f_high].
    Returns: DataFrame (freq_Hz, valor, prominencia) ordenat per prominència."""
    mask = (freqs >= f_low) & (freqs <= f_high)
    f, s = freqs[mask], spectrum[mask]
    if len(f) < 3:
        return pd.DataFrame(columns=['freq_Hz', 'valor', 'prominencia'])
    df_hz = f[1] - f[0]
    dist = max(1, int(round(min_distance_hz / df_hz)))
    idx, props = signal.find_peaks(s, distance=dist, prominence=0)
    if len(idx) == 0:
        return pd.DataFrame(columns=['freq_Hz', 'valor', 'prominencia'])
    order = np.argsort(props['prominences'])[::-1][:n_peaks]
    out = pd.DataFrame({'freq_Hz': f[idx[order]], 'valor': s[idx[order]],
                        'prominencia': props['prominences'][order]})
    return out.reset_index(drop=True)


def freq_band_energy(freqs, psd, low_pct=0.05, high_pct=0.95, f_low=F_MIN, f_high=F_MAX):
    """Banda freqüencial que conté entre el 5 % i el 95 % de l'energia (PSD acumulada)."""
    mask = (freqs >= f_low) & (freqs <= f_high)
    f, p = freqs[mask], psd[mask]
    cum = np.concatenate([[0], np.cumsum(0.5 * (p[1:] + p[:-1]) * np.diff(f))])
    cum /= cum[-1] if cum[-1] > 0 else 1
    return float(np.interp(low_pct, cum, f)), float(np.interp(high_pct, cum, f))


print("✓ Funcions d'anàlisi definides")

## 📂 5 - Càrrega de dades

In [ ]:
# ⬇️ CONFIGURACIÓ DEL SENSOR (ha de coincidir amb la configuració del MEMS Studio) ⬇️
ACC_FS_G     = 4      # Fons d'escala acceleròmetre [g]   → opcions ASM330: 2, 4, 8, 16
GYRO_FS_DPS  = 2000   # Fons d'escala giroscopi [dps]     → opcions ASM330: 125, 250, 500, 1000, 2000, 4000
SAT_MARGIN   = 0.98   # Fracció del fons d'escala a partir de la qual es considera saturat (98 %)
SAT_GAP_S    = 0.05   # s · mostres saturades separades menys d'això s'agrupen en un mateix episodi
PLOT_STEP    = 1      # 1 = totes les mostres a la gràfica; 2, 4… per alleugerir fitxers molt llargs

print(f"→ Carregant: {CSV_FILENAME}")
df = pd.read_csv(CSV_FILENAME, skiprows=1)
print(f"✓ CSV carregat: {len(df)} mostres")

time_col = 'time[us]'
acc_cols = ['acc_x[mg]', 'acc_y[mg]', 'acc_z[mg]']
gyro_cols = ['gyro_x[mdps]', 'gyro_y[mdps]', 'gyro_z[mdps]']
falten = [c for c in [time_col] + acc_cols + gyro_cols if c not in df.columns]
if falten:
    raise KeyError(f"❌ Falten columnes al CSV: {falten}")
df = df.dropna(subset=[time_col] + acc_cols + gyro_cols).reset_index(drop=True)

# Acceleròmetre (mg)
ax = df['acc_x[mg]'].values.astype(float)
ay = df['acc_y[mg]'].values.astype(float)
az = df['acc_z[mg]'].values.astype(float)

# Giroscopi mdps → dps
gx_raw = df['gyro_x[mdps]'].values * MDPS_TO_DPS
gy_raw = df['gyro_y[mdps]'].values * MDPS_TO_DPS
gz_raw = df['gyro_z[mdps]'].values * MDPS_TO_DPS

# Remapejat del giroscopi (igual que al notebook del filtre complementari)
gx = gy_raw   # Roll
gy = gx_raw   # Pitch
gz = gz_raw   # Yaw

# Vector temps: µs → s
time_us = df[time_col].values.astype(float)
time = (time_us - time_us[0]) / 1_000_000.0
dt_all = np.diff(time)
dt = np.median(dt_all)
FS = 1.0 / np.mean(dt_all)
NYQUIST = FS / 2.0

print(f"\n✓ Dades processades:")
print(f"  - Duració total:      {time[-1]:.2f} s")
print(f"  - Mostres:            {len(ax)}")
print(f"  - Sample rate real:   {FS:.2f} Hz")
print(f"  - Nyquist:            {NYQUIST:.1f} Hz")
print(f"  - Jitter Δt (std):    {np.std(dt_all)*1e6:.1f} µs")
if 'temp[C]' in df.columns:
    print(f"  - Temperatura:        {df['temp[C]'].mean():.1f} °C "
          f"(min {df['temp[C]'].min():.1f} / màx {df['temp[C]'].max():.1f})")

if F_MAX >= NYQUIST:
    print(f"\n⚠️ F_MAX ({F_MAX} Hz) ≥ Nyquist ({NYQUIST:.1f} Hz) → s'ajusta a {NYQUIST - 5:.1f} Hz")
    F_MAX = NYQUIST - 5.0
if np.std(dt_all) > 0.05 * dt:
    print("⚠️ Mostreig irregular (jitter > 5 %): la FFT assumeix Δt constant")

# ====================================================================
# DETECCIÓ DE SATURACIÓ
# ====================================================================
print("\n" + "=" * 60)
print(f"DETECCIÓ DE SATURACIÓ  (acc ±{ACC_FS_G} g · gyro ±{GYRO_FS_DPS} dps · llindar {SAT_MARGIN*100:.0f} %)")
print("=" * 60)

ACC_SAT_MG   = ACC_FS_G * 1000.0 * SAT_MARGIN
GYRO_SAT_DPS = GYRO_FS_DPS * SAT_MARGIN

# Canals tal com els veu el sensor (abans de remapejar/invertir) → noms coincideixen amb el CSV
canals_sat = {
    'acc_x':  (ax,     ACC_SAT_MG,   'mg'),
    'acc_y':  (ay,     ACC_SAT_MG,   'mg'),
    'acc_z':  (az,     ACC_SAT_MG,   'mg'),
    'gyro_x': (gx_raw, GYRO_SAT_DPS, 'dps'),
    'gyro_y': (gy_raw, GYRO_SAT_DPS, 'dps'),
    'gyro_z': (gz_raw, GYRO_SAT_DPS, 'dps'),
}
SAT_MASK = {k: np.abs(v) >= lim for k, (v, lim, _) in canals_sat.items()}

print(f"{'Canal':<8}{'Pic |x|':>12}{'Fons esc.':>12}{'% FS':>8}{'Mostres sat.':>14}{'% temps':>10}")
for k, (v, lim, u) in canals_sat.items():
    fs_val = ACC_FS_G * 1000.0 if u == 'mg' else GYRO_FS_DPS
    pk = np.max(np.abs(v))
    n = int(SAT_MASK[k].sum())
    flag = '  ❌' if n else ('  ⚠️' if pk > 0.8 * fs_val else '  ✅')
    print(f"{k:<8}{pk:>9.1f} {u:<3}{fs_val:>8.0f} {u:<3}{pk/fs_val*100:>7.1f}%{n:>14d}{n/len(v)*100:>9.3f}%{flag}")

sat_any = np.zeros(len(time), dtype=bool)
for m in SAT_MASK.values():
    sat_any |= m

# Agrupa mostres saturades en episodis
SAT_EPISODES = []
idx = np.flatnonzero(sat_any)
if idx.size:
    gap = int(np.ceil(SAT_GAP_S / dt))
    talls = np.flatnonzero(np.diff(idx) > gap)
    for s, e in zip(np.r_[0, talls + 1], np.r_[talls, idx.size - 1]):
        i0, i1 = idx[s], idx[e]
        eixos = [k for k, m in SAT_MASK.items() if m[i0:i1 + 1].any()]
        SAT_EPISODES.append({'t_inici_s': time[i0], 't_final_s': time[i1],
                             'durada_ms': (time[i1] - time[i0]) * 1000 + dt * 1000,
                             'mostres': int(sat_any[i0:i1 + 1].sum()), 'canals': ', '.join(eixos)})
SAT_EPISODES = pd.DataFrame(SAT_EPISODES)

if SAT_EPISODES.empty:
    print("\n✅ Cap mostra saturada → els pics d'impacte són fiables")
else:
    print(f"\n❌ {len(SAT_EPISODES)} episodis de saturació "
          f"({int(sat_any.sum())} mostres, {sat_any.mean()*100:.3f} % del temps):")
    print(SAT_EPISODES.head(30).to_string(index=False, float_format=lambda x: f"{x:.3f}"))
    if len(SAT_EPISODES) > 30:
        print(f"   … i {len(SAT_EPISODES) - 30} episodis més (variable SAT_EPISODES)")
    print("\n⚠️ Implicacions:")
    print("   • Pics reals > fons d'escala → kurtosi, factor de cresta i acceleració màxima SUBESTIMATS")
    print("   • El retall genera harmònics artificials d'alta freqüència a la FFT/PSD")
    seguent = {2: 4, 4: 8, 8: 16}.get(ACC_FS_G)
    if any(SAT_MASK[k].any() for k in ('acc_x', 'acc_y', 'acc_z')) and seguent:
        print(f"   → Recomanació: pròxima captura amb acc a ±{seguent} g (ACC_FS_G = {seguent})")

# ====================================================================
# GRÀFICA TEMPORAL (eixos tal com surten del sensor, abans de corregir Z)
# ====================================================================
sl = slice(None, None, PLOT_STEP)
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.07,
                    subplot_titles=(f"Acceleròmetre [mg] · fons d'escala ±{ACC_FS_G} g",
                                    f"Giroscopi [dps] · fons d'escala ±{GYRO_FS_DPS} dps"))
for row, trios in ((1, [('acc_x', ax, 'x'), ('acc_y', ay, 'y'), ('acc_z', az, 'z')]),
                   (2, [('gyro_x', gx_raw, 'x'), ('gyro_y', gy_raw, 'y'), ('gyro_z', gz_raw, 'z')])):
    for name, v, c in trios:
        fig.add_trace(go.Scattergl(x=time[sl], y=v[sl], mode='lines', name=name,
                                   line=dict(color=COLORS[c], width=1)), row=row, col=1)
        m = SAT_MASK[name]
        if m.any():
            fig.add_trace(go.Scattergl(x=time[m], y=v[m], mode='markers', name=f'{name} saturat',
                                       marker=dict(color='red', size=6, symbol='x'),
                                       showlegend=False), row=row, col=1)
    lim = ACC_FS_G * 1000.0 if row == 1 else GYRO_FS_DPS
    for s in (1, -1):
        fig.add_hline(y=s * lim, line=dict(color='red', dash='dash', width=1.5), row=row, col=1)

for _, ep in SAT_EPISODES.head(50).iterrows():
    fig.add_vrect(x0=ep['t_inici_s'] - 0.05, x1=ep['t_final_s'] + 0.05,
                  fillcolor='red', opacity=0.15, line_width=0)

fig.update_yaxes(title_text='mg', row=1, col=1)
fig.update_yaxes(title_text='dps', row=2, col=1)
fig.update_xaxes(title_text='Temps [s]', row=2, col=1)
fig.update_layout(template=PLOTLY_TEMPLATE, height=750, hovermode='x unified',
                  title="📈 Senyal temporal brut · ✖ vermelles = mostres saturades · franges = episodis")
fig.show()

print(f"\n✅ Dades carregades correctament!")

## 🎯 6 - Calibració + Correcció de l'eix Z

1. **Bias del giroscopi** a partir dels primers `CALIBRATION_TIME` segons (sensor quiet).
2. Es **descarta** el període de calibració (no és vibració de conducció).
3. ⚠️ **Eix Z invertit al muntatge:** `az = −az` i `gz = −gz`.
4. S'elimina la component **DC** de cada canal (gravetat + offsets) → només queda vibració.

In [ ]:
print("=" * 60)
print("CALIBRACIÓ I CORRECCIÓ D'EIXOS")
print("=" * 60)

calib_end_idx = np.searchsorted(time, CALIBRATION_TIME)
if calib_end_idx < 100:
    print(f"⚠️ Només {calib_end_idx} mostres de calibració (mínim recomanat: 100)")

# 1) Bias del giroscopi
gx_bias = np.mean(gx[:calib_end_idx])
gy_bias = np.mean(gy[:calib_end_idx])
gz_bias = np.mean(gz[:calib_end_idx])
acc_std_cal = np.sqrt(np.std(ax[:calib_end_idx])**2 + np.std(ay[:calib_end_idx])**2 + np.std(az[:calib_end_idx])**2)
print(f"→ Període de calibració: 0.00 – {time[calib_end_idx]:.2f} s ({calib_end_idx} mostres)")
print(f"→ Bias gyro: gx={gx_bias:+.3f}  gy={gy_bias:+.3f}  gz={gz_bias:+.3f} dps")
print(f"→ Estabilitat acc durant calibració: std={acc_std_cal:.1f} mg"
      + ("  ✅" if acc_std_cal < 30 else "  ⚠️ hi ha moviment!"))

# 2) Descarta el període de calibració
sl = slice(calib_end_idx, None)
t = time[sl] - time[calib_end_idx]
ax_c, ay_c, az_c = ax[sl], ay[sl], az[sl]
gx_c, gy_c, gz_c = gx[sl] - gx_bias, gy[sl] - gy_bias, gz[sl] - gz_bias

# 3) Correcció de l'eix Z invertit
az_c = -az_c
gz_c = -gz_c
print("✓ Eix Z invertit corregit (az = −az, gz = −gz)")

# Conversió a g
ax_g, ay_g, az_g = ax_c * MG_TO_G, ay_c * MG_TO_G, az_c * MG_TO_G
print(f"→ Mitjanes (gravetat + offsets): ax={np.mean(ax_g):+.3f} g  ay={np.mean(ay_g):+.3f} g  az={np.mean(az_g):+.3f} g")
print(f"→ |g| mitjà = {np.sqrt(np.mean(ax_g)**2 + np.mean(ay_g)**2 + np.mean(az_g)**2):.3f} g (ideal ≈ 1.000)")

# 4) Elimina DC → senyals de vibració
ax_vib = ax_g - np.mean(ax_g)
ay_vib = ay_g - np.mean(ay_g)
az_vib = az_g - np.mean(az_g)
gx_vib = gx_c - np.mean(gx_c)
gy_vib = gy_c - np.mean(gy_c)
gz_vib = gz_c - np.mean(gz_c)

# Magnitud del vector de vibració
a_total_vib = np.sqrt(ax_vib**2 + ay_vib**2 + az_vib**2)

DURATION = t[-1]
ACC = {'x': ax_vib, 'y': ay_vib, 'z': az_vib}     # [g]
GYR = {'x': gx_vib, 'y': gy_vib, 'z': gz_vib}     # [dps]

print(f"\n✓ Senyal útil: {DURATION:.2f} s · {len(t)} mostres")
if len(t) < NPERSEG:
    print(f"⚠️ Menys mostres ({len(t)}) que NPERSEG ({NPERSEG}) → es reduirà el segment de Welch")
n_avg = max(1, int((len(t) - NPERSEG) / (NPERSEG * (1 - WELCH_OVERLAP))) + 1)
print(f"→ Promitjos de Welch ≈ {n_avg} · Δf = {FS/NPERSEG:.3f} Hz")
print("=" * 60)

## 📈 7 - Anàlisi FFT (espectre d'amplitud)

FFT de tot el registre útil amb finestra de Hann → **amplitud de pic** (g per a l'acceleròmetre, dps per al giroscopi). Màxima resolució freqüencial (Δf = 1/T) i útil per identificar **freqüències discretes** (rotació del motor, cadència, engranatges). Per a nivells d'energia s'ha d'usar la PSD (secció 8).

In [ ]:
FFT_ACC, FFT_GYR, PEAKS_FFT_ACC, PEAKS_FFT_GYR = {}, {}, {}, {}
for k in 'xyz':
    FFT_ACC[k] = compute_fft_onesided(ACC[k], FS)
    FFT_GYR[k] = compute_fft_onesided(GYR[k], FS)
    PEAKS_FFT_ACC[k] = find_spectral_peaks(*FFT_ACC[k], n_peaks=N_PEAKS)
    PEAKS_FFT_GYR[k] = find_spectral_peaks(*FFT_GYR[k], n_peaks=N_PEAKS)

print(f"Resolució FFT: Δf = {FFT_ACC['x'][0][1]:.4f} Hz")

fig = make_subplots(rows=2, cols=1, shared_xaxes=False, vertical_spacing=0.12,
                    subplot_titles=('Acceleròmetre — Amplitud FFT [g pic]',
                                    'Giroscopi — Amplitud FFT [dps pic]'))
for row, (data, peaks, unit) in enumerate([(FFT_ACC, PEAKS_FFT_ACC, 'g'),
                                           (FFT_GYR, PEAKS_FFT_GYR, 'dps')], start=1):
    for k in 'xyz':
        f, a = data[k]
        m = (f >= F_MIN) & (f <= F_MAX)
        fig.add_trace(go.Scatter(x=f[m], y=a[m], mode='lines', line=dict(color=COLORS[k], width=1),
                                 name=f"{'acc' if row == 1 else 'gyro'}_{k}",
                                 legendgroup=f'{row}{k}',
                                 hovertemplate='%{x:.2f} Hz<br>%{y:.4f} ' + unit), row=row, col=1)
        top5 = peaks[k].head(5)
        fig.add_trace(go.Scatter(x=top5['freq_Hz'], y=top5['valor'], mode='markers+text',
                                 marker=dict(color=COLORS[k], size=8, symbol='triangle-down'),
                                 text=[f"{v:.1f}" for v in top5['freq_Hz']], textposition='top center',
                                 textfont=dict(size=9, color=COLORS[k]), showlegend=False,
                                 legendgroup=f'{row}{k}',
                                 hovertemplate='Pic %{x:.2f} Hz<br>%{y:.4f} ' + unit), row=row, col=1)
for r, u in [(1, 'Amplitud [g]'), (2, 'Amplitud [dps]')]:
    fig.update_xaxes(title_text='Freqüència [Hz]', range=[0, F_MAX], row=r, col=1)
    fig.update_yaxes(title_text=u, row=r, col=1)
fig.update_layout(template=PLOTLY_TEMPLATE, height=850,
                  title='📈 FFT — Espectre d\'amplitud (top 5 pics per canal)', hovermode='x unified')
fig.show()

print("=" * 60)
print("TOP 5 PICS FFT PER CANAL")
print("=" * 60)
for name, peaks, unit in [('acc', PEAKS_FFT_ACC, 'g'), ('gyro', PEAKS_FFT_GYR, 'dps')]:
    for k in 'xyz':
        s = ", ".join(f"{r.freq_Hz:6.2f} Hz ({r.valor:.4f} {unit})" for r in peaks[k].head(5).itertuples())
        print(f"{name}_{k}: {s}")

## 📊 8 - PSD de Welch (densitat espectral de potència)

La **PSD** (g²/Hz) és la magnitud estàndard per especificar vibració aleatòria en bancs d'assaig (IEC 60068-2-64, MIL-STD-810, ISO 16750-3). L'àrea sota la corba = g-RMS².
Les línies verticals grises marquen els centres de banda de 1/3 d'octava.

In [ ]:
PSD_ACC, PSD_GYR, PEAKS_PSD_ACC = {}, {}, {}
for k in 'xyz':
    PSD_ACC[k] = compute_psd_welch(ACC[k], FS)
    PSD_GYR[k] = compute_psd_welch(GYR[k], FS)
    PEAKS_PSD_ACC[k] = find_spectral_peaks(*PSD_ACC[k], n_peaks=N_PEAKS)
f_psd = PSD_ACC['x'][0]
# PSD de la "suma vectorial" (suma d'energies dels 3 eixos)
PSD_ACC_SUM = PSD_ACC['x'][1] + PSD_ACC['y'][1] + PSD_ACC['z'][1]
print(f"Resolució PSD: Δf = {f_psd[1]:.3f} Hz")
print(f"⚠️ Per sota de ≈{4*f_psd[1]:.1f} Hz la PSD té pocs bins i el detrend la pot atenuar")

fig = make_subplots(rows=2, cols=1, vertical_spacing=0.12,
                    subplot_titles=('Acceleròmetre — PSD [g²/Hz]', 'Giroscopi — PSD [dps²/Hz]'))
for row, (data, unit) in enumerate([(PSD_ACC, 'g²/Hz'), (PSD_GYR, 'dps²/Hz')], start=1):
    for k in 'xyz':
        f, p = data[k]
        m = (f >= F_MIN) & (f <= F_MAX)
        fig.add_trace(go.Scatter(x=f[m], y=p[m], mode='lines', line=dict(color=COLORS[k], width=1.3),
                                 name=f"{'acc' if row == 1 else 'gyro'}_{k}",
                                 hovertemplate='%{x:.2f} Hz<br>%{y:.3e} ' + unit), row=row, col=1)
        pk = find_spectral_peaks(f, p, n_peaks=3)
        fig.add_trace(go.Scatter(x=pk['freq_Hz'], y=pk['valor'], mode='markers+text',
                                 marker=dict(color=COLORS[k], size=8, symbol='triangle-down'),
                                 text=[f"{v:.1f} Hz" for v in pk['freq_Hz']], textposition='top center',
                                 textfont=dict(size=9, color=COLORS[k]), showlegend=False), row=row, col=1)
    for fc in OCTAVE_CENTERS[(OCTAVE_CENTERS >= F_MIN) & (OCTAVE_CENTERS <= F_MAX)]:
        fig.add_vline(x=fc, line=dict(color='lightgray', width=0.7, dash='dot'), row=row, col=1)
    fig.update_xaxes(type='log', title_text='Freqüència [Hz]', range=[np.log10(F_MIN), np.log10(F_MAX)],
                     row=row, col=1)
    fig.update_yaxes(type='log', title_text=f'PSD [{unit}]', exponentformat='power', row=row, col=1)
fig.update_layout(template=PLOTLY_TEMPLATE, height=850,
                  title='📊 PSD de Welch (escala log-log) — top 3 pics per canal')
fig.show()

print("=" * 60)
print("PICS DOMINANTS PSD (acceleròmetre)")
print("=" * 60)
for k in 'xyz':
    s = ", ".join(f"{r.freq_Hz:6.2f} Hz ({r.valor:.2e} g²/Hz)" for r in PEAKS_PSD_ACC[k].head(5).itertuples())
    print(f"acc_{k}: {s}")

## 🎚️ 9 - Anàlisi per bandes de 1/3 d'octava (ISO 266)

g-RMS de cada banda: √∫PSD df entre f_low = fc/2^(1/6) i f_high = fc·2^(1/6).
La **PSD mitjana per banda** (g²/Hz) és directament utilitzable com a punts de ruptura (*breakpoints*) d'un perfil ASD al controlador del banc.

In [ ]:
OCT = {k: compute_octave_band_rms(f_psd, PSD_ACC[k][1], OCTAVE_CENTERS) for k in 'xyz'}
oct_table = pd.DataFrame({
    'fc [Hz]': OCT['x']['fc_Hz'],
    'f_low [Hz]': OCT['x']['f_low_Hz'].round(2),
    'f_high [Hz]': OCT['x']['f_high_Hz'].round(2),
    'rms_x [g]': OCT['x']['rms'],
    'rms_y [g]': OCT['y']['rms'],
    'rms_z [g]': OCT['z']['rms'],
    'PSD_x [g²/Hz]': OCT['x']['psd_mitjana'],
    'PSD_y [g²/Hz]': OCT['y']['psd_mitjana'],
    'PSD_z [g²/Hz]': OCT['z']['psd_mitjana'],
})

labels = [f"{fc:g}" for fc in OCTAVE_CENTERS]
fig = go.Figure()
for k in 'xyz':
    fig.add_trace(go.Bar(x=labels, y=OCT[k]['rms'], name=f'acc_{k}', marker_color=COLORS[k],
                         hovertemplate='fc=%{x} Hz<br>%{y:.4f} g RMS'))
fig.update_layout(template=PLOTLY_TEMPLATE, barmode='group', height=500,
                  title='🎚️ g-RMS per banda de 1/3 d\'octava',
                  xaxis_title='Freqüència central de banda [Hz]', yaxis_title='g-RMS [g]')
fig.show()

print("=" * 60)
print("TAULA 1/3 D'OCTAVA — ACCELERÒMETRE")
print("=" * 60)
print(oct_table.to_string(index=False, float_format=lambda v: f"{v:.4g}"))
trunc = OCT['x'][OCT['x']['truncada_nyquist']]['fc_Hz'].tolist()
if trunc:
    print(f"\n⚠️ Bandes truncades per Nyquist ({NYQUIST:.1f} Hz): {trunc} Hz")
low = OCTAVE_CENTERS[(OCTAVE_CENTERS / 2**(1/6)) * (2**(1/3) - 1) < 2 * f_psd[1]]
if len(low):
    print(f"⚠️ Bandes amb < 2 bins de PSD (poc fiables): {low.tolist()} Hz")
for k in 'xyz':
    i = OCT[k]['rms'].idxmax()
    print(f"→ Banda dominant acc_{k}: {OCT[k]['fc_Hz'][i]:g} Hz ({OCT[k]['rms'][i]:.4f} g RMS)")

## 🌈 10 - Espectrograma temps-freqüència

Mostra **quan** apareixen les vibracions (canvis de terreny, acceleracions del motor, frenades). Escala en dB re 1 g²/Hz.

In [ ]:
nps = int(min(NPERSEG_SPECTROGRAM, len(t) // 4))
nov = int(nps * WELCH_OVERLAP)
SPEC = {}
for k in 'xyz':
    f_s, t_s, Sxx = signal.spectrogram(ACC[k], fs=FS, window=WINDOW, nperseg=nps, noverlap=nov,
                                       detrend='linear', scaling='density')
    m = (f_s >= F_MIN) & (f_s <= F_MAX)
    SPEC[k] = (f_s[m], t_s, 10 * np.log10(Sxx[m] + 1e-30))

all_db = np.concatenate([SPEC[k][2].ravel() for k in 'xyz'])
zmin, zmax = np.percentile(all_db, 5), np.percentile(all_db, 99.5)

fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.06,
                    subplot_titles=[f'acc_{k}' for k in 'xyz'])
for i, k in enumerate('xyz', start=1):
    f_s, t_s, db = SPEC[k]
    fig.add_trace(go.Heatmap(x=t_s, y=f_s, z=db, colorscale='Turbo', zmin=zmin, zmax=zmax,
                             showscale=(i == 1), colorbar=dict(title='dB re g²/Hz'),
                             hovertemplate='t=%{x:.1f} s<br>f=%{y:.1f} Hz<br>%{z:.1f} dB<extra></extra>'),
                  row=i, col=1)
    fig.update_yaxes(title_text='Freq. [Hz]', row=i, col=1)
fig.update_xaxes(title_text='Temps [s]', row=3, col=1)
fig.update_layout(template=PLOTLY_TEMPLATE, height=900,
                  title=f'🌈 Espectrograma (nperseg={nps}, Δt≈{(nps-nov)/FS:.2f} s, Δf≈{FS/nps:.2f} Hz)')
fig.show()

# Instant de màxima energia
for k in 'xyz':
    f_s, t_s, db = SPEC[k]
    energy_t = np.sum(10**(db / 10), axis=0)
    print(f"→ acc_{k}: màxima energia a t ≈ {t_s[np.argmax(energy_t)]:.1f} s")

## 📐 11 - Estadístiques vibratòries globals

- **g-RMS** per integració de la PSD (coherent amb la representació espectral).
- **Curtosi** = E[(x−μ)⁴]/σ⁴ → **≈ 3 per a un senyal gaussià**; valors més alts indiquen un senyal **més impulsiu** (cops, sotracs). Si la curtosi > 3–4, un assaig aleatori gaussià pur **subestima** el dany: valorar un perfil *kurtosis control* o afegir xocs.
- **Factor de cresta** = |pic| / RMS (gaussià ≈ 3–4,5).

In [ ]:
def channel_stats(name, x, freqs, psd, unit, to_ms2=False):
    rms_t = np.sqrt(np.mean(x**2))
    peak = np.max(np.abs(x))
    row = {'canal': name, 'unitat': unit,
           f'RMS total {F_MIN:g}-{F_MAX:g} Hz (PSD)': compute_grms(freqs, psd, F_MIN, F_MAX),
           'RMS temporal': rms_t}
    for lo, hi in GRMS_BANDS:
        row[f'RMS {lo:g}-{hi:g} Hz'] = compute_grms(freqs, psd, lo, min(hi, F_MAX))
    row['Curtosi'] = stats.kurtosis(x, fisher=False)
    row['Factor de cresta'] = peak / rms_t if rms_t > 0 else np.nan
    row['Pic |x|'] = peak
    row['Pic [m/s²]'] = peak * G_TO_MS2 if to_ms2 else np.nan
    return row

rows = [channel_stats(f'acc_{k}', ACC[k], *PSD_ACC[k], 'g', True) for k in 'xyz']

# Suma vectorial: RMS espectral = √(Σ PSD); pics/curtosi a partir de la magnitud temporal
tot = channel_stats('acc_total (vector)', a_total_vib, f_psd, PSD_ACC_SUM, 'g', True)
tot['RMS temporal'] = np.sqrt(np.mean(a_total_vib**2))
tot['Curtosi'] = np.nan   # la magnitud no és de mitjana zero → la curtosi no és comparable
tot['Factor de cresta'] = np.max(a_total_vib) / tot['RMS temporal']
rows.append(tot)
rows += [channel_stats(f'gyro_{k}', GYR[k], *PSD_GYR[k], 'dps') for k in 'xyz']

STATS = pd.DataFrame(rows).set_index('canal')

print("=" * 60)
print("ESTADÍSTIQUES VIBRATÒRIES GLOBALS")
print("=" * 60)
print(STATS.to_string(float_format=lambda v: f"{v:.4f}"))
print("\nℹ️ Curtosi: gaussià ≈ 3 · >3 = més impulsiu (sotracs/cops)")
for k in 'xyz':
    ku = STATS.loc[f'acc_{k}', 'Curtosi']
    if ku > 4:
        print(f"⚠️ acc_{k}: curtosi {ku:.2f} → senyal impulsiu, un perfil gaussià pur subestimaria el dany")

## 🔥 12 - Fatigue Damage Spectrum (FDS)

El FDS indica el **dany per fatiga relatiu** que la vibració mesurada produiria en un component amb freqüència pròpia *fₙ* (1 GDL, ζ = 5 %, b = 6).
Permet identificar **quines freqüències són més perilloses** per als components del motor (suports, PCB, connectors, imants) i validar que un perfil d'assaig accelerat produeix un dany ≥ al real.

> ⚠️ **FDS normalitzat:** les corbes es normalitzen al màxim global (suma vectorial = 1). Els valors són **relatius**: no inclouen la constant de material (K, C de la corba S-N), per tant només serveixen per comparar freqüències, eixos o perfils.

In [ ]:
FN = np.geomspace(F_MIN, min(F_MAX, NYQUIST - 5.0), FDS_N_POINTS)
FDS = {}
for k in 'xyz':
    FDS[k], _ = compute_fds(PSD_ACC[k][1], f_psd, FN, FDS_DAMPING, DURATION, FDS_B)
FDS['total'], _ = compute_fds(PSD_ACC_SUM, f_psd, FN, FDS_DAMPING, DURATION, FDS_B)
fds_ref = np.max(FDS['total'])
FDS_NORM = {k: v / fds_ref for k, v in FDS.items()}
FDS_CRIT = {k: FN[np.argmax(v)] for k, v in FDS.items()}

fig = go.Figure()
for k, lbl in [('x', 'acc_x'), ('y', 'acc_y'), ('z', 'acc_z'), ('total', 'Suma vectorial')]:
    fig.add_trace(go.Scatter(x=FN, y=FDS_NORM[k], mode='lines', name=lbl,
                             line=dict(color=COLORS[k], width=3 if k == 'total' else 1.8,
                                       dash='dash' if k == 'total' else 'solid'),
                             hovertemplate='fn=%{x:.2f} Hz<br>D=%{y:.3e}'))
i_crit = np.argmax(FDS_NORM['total'])
fig.add_annotation(x=np.log10(FN[i_crit]), y=np.log10(FDS_NORM['total'][i_crit]),
                   text=f"⚠️ fₙ crítica: {FN[i_crit]:.1f} Hz", showarrow=True, arrowhead=2,
                   ax=40, ay=-40, font=dict(color='red', size=12))
fig.update_xaxes(type='log', title_text='Freqüència natural fₙ [Hz]')
fig.update_yaxes(type='log', title_text='Dany relatiu (normalitzat)', exponentformat='power',
                 range=[np.log10(max(1e-8, np.min(FDS_NORM['total'][FDS_NORM['total'] > 0]))), 0.3])
fig.update_layout(template=PLOTLY_TEMPLATE, height=550,
                  title=f'🔥 Fatigue Damage Spectrum (b={FDS_B}, ζ={FDS_DAMPING}, T={DURATION:.1f} s)')
fig.show()

print("=" * 60)
print("FREQÜÈNCIES CRÍTIQUES (FDS)")
print("=" * 60)
for k, lbl in [('x', 'acc_x'), ('y', 'acc_y'), ('z', 'acc_z'), ('total', 'Suma vectorial')]:
    pk = find_spectral_peaks(FN, FDS_NORM[k], n_peaks=3, f_low=FN[0], f_high=FN[-1], min_distance_hz=0)
    others = ", ".join(f"{f:.1f} Hz" for f in pk['freq_Hz'])
    print(f"{lbl:15s}: màx a {FDS_CRIT[k]:6.1f} Hz · pics locals: {others}")
print("\n⚠️ Valors relatius (normalitzats al màxim de la suma vectorial)")

## 🧾 13 - Taula resum per configurar el banc de vibració

**Perfil recomanat:** vibració aleatòria (*random*) amb la forma de la PSD mesurada (punts de 1/3 d'octava), a cada eix, al nivell g-RMS mesurat.
**Envelliment accelerat** (Basquin/Miner): per reproduir `HORES_US_REAL` en `HORES_ASSAIG`, el nivell s'escala per

$$ k = \left(\frac{T_{real}}{T_{assaig}}\right)^{1/b} \quad\Rightarrow\quad G_{assaig} = k \cdot G_{real},\quad PSD_{assaig} = k^2 \cdot PSD_{real} $$

> ⚠️ Un sol registre de ~30–40 s **no és representatiu** de tota la vida d'ús: cal combinar registres de diferents terrenys/velocitats (perfil de missió) abans de fixar el perfil definitiu. No s'ha de superar el límit elàstic ni excitar modes no presents en ús real (factor k recomanat ≤ 2).

In [ ]:
k_acc = (HORES_US_REAL / HORES_ASSAIG) ** (1.0 / FDS_B)

summary = []
add = lambda param, val, unitat='': summary.append({'Paràmetre': param, 'Valor': val, 'Unitat': unitat})
add('Posició del sensor', SENSOR_POSITION)
add('Fitxer', CSV_FILENAME)
add('Duració analitzada', f"{DURATION:.2f}", 's')
add('Freqüència de mostreig', f"{FS:.2f}", 'Hz')
add('Rang analitzat', f"{F_MIN:g} – {F_MAX:g}", 'Hz')
for k in 'xyz':
    add(f'g-RMS acc_{k}', f"{STATS.loc[f'acc_{k}', f'RMS total {F_MIN:g}-{F_MAX:g} Hz (PSD)']:.4f}", 'g')
add('g-RMS vectorial', f"{STATS.loc['acc_total (vector)', f'RMS total {F_MIN:g}-{F_MAX:g} Hz (PSD)']:.4f}", 'g')
for k in 'xyz':
    top3 = PEAKS_FFT_ACC[k].head(3)['freq_Hz']
    add(f'Freq. dominants acc_{k} (FFT top 3)', ", ".join(f"{f:.1f}" for f in top3), 'Hz')
for k in 'xyz':
    add(f'Curtosi acc_{k}', f"{STATS.loc[f'acc_{k}', 'Curtosi']:.2f}", '-')
for k in 'xyz':
    add(f'Factor de cresta acc_{k}', f"{STATS.loc[f'acc_{k}', 'Factor de cresta']:.2f}", '-')
for k, lbl in [('x', 'acc_x'), ('y', 'acc_y'), ('z', 'acc_z'), ('total', 'vectorial')]:
    add(f'Freq. crítica FDS {lbl}', f"{FDS_CRIT[k]:.1f}", 'Hz')

# Perfil recomanat
for k in 'xyz':
    f5, f95 = freq_band_energy(f_psd, PSD_ACC[k][1])
    g_real = STATS.loc[f'acc_{k}', f'RMS total {F_MIN:g}-{F_MAX:g} Hz (PSD)']
    add(f'Perfil acc_{k}: banda 90 % energia', f"{f5:.1f} – {f95:.1f}", 'Hz')
    add(f'Perfil acc_{k}: g-RMS real', f"{g_real:.4f}", 'g')
    add(f'Perfil acc_{k}: g-RMS accelerat (k={k_acc:.2f})', f"{g_real * k_acc:.4f}", 'g')
add('Factor d\'acceleració k', f"{k_acc:.3f}", f'({HORES_US_REAL:g} h → {HORES_ASSAIG:g} h, b={FDS_B:g})')
add('Tipus d\'assaig recomanat', 'Random (ASD 1/3 octava) per eix'
    + (' + kurtosis control / xocs' if max(STATS.loc[f'acc_{k}', 'Curtosi'] for k in 'xyz') > 4 else ''))

SUMMARY = pd.DataFrame(summary)

# Perfil ASD per al controlador del banc (PSD mitjana per banda de 1/3 d'octava)
ASD_PROFILE = pd.DataFrame({'fc [Hz]': OCTAVE_CENTERS})
for k in 'xyz':
    ASD_PROFILE[f'ASD_real_{k} [g²/Hz]'] = OCT[k]['psd_mitjana'].values
    ASD_PROFILE[f'ASD_assaig_{k} [g²/Hz]'] = OCT[k]['psd_mitjana'].values * k_acc**2
ASD_PROFILE = ASD_PROFILE[(ASD_PROFILE['fc [Hz]'] >= F_MIN) & (ASD_PROFILE['fc [Hz]'] <= F_MAX)]

print("=" * 60)
print("🧾 RESUM PER AL BANC DE VIBRACIÓ")
print("=" * 60)
print(SUMMARY.to_string(index=False))
if k_acc > 2:
    print(f"\n⚠️ k = {k_acc:.2f} > 2: risc de modes de fallada no representatius (plasticitat). "
          "Allarga HORES_ASSAIG o valida-ho amb el fabricant del material.")
print("\n" + "=" * 60)
print("PERFIL ASD (1/3 octava) PER AL CONTROLADOR")
print("=" * 60)
print(ASD_PROFILE.to_string(index=False, float_format=lambda v: f"{v:.3e}"))

## 💾 14 - Exportació de resultats

In [ ]:
base = os.path.splitext(os.path.basename(CSV_FILENAME))[0]
out_files = {
    f'{base}_resum_vibracio.csv': SUMMARY,
    f'{base}_estadistiques.csv': STATS.reset_index(),
    f'{base}_terc_octava.csv': oct_table,
    f'{base}_perfil_ASD.csv': ASD_PROFILE,
    f'{base}_PSD.csv': pd.DataFrame({'freq [Hz]': f_psd,
                                     **{f'PSD_acc_{k} [g²/Hz]': PSD_ACC[k][1] for k in 'xyz'},
                                     **{f'PSD_gyro_{k} [dps²/Hz]': PSD_GYR[k][1] for k in 'xyz'}}),
    f'{base}_FDS.csv': pd.DataFrame({'fn [Hz]': FN, **{f'FDS_norm_{k}': FDS_NORM[k] for k in FDS_NORM}}),
}
print("=" * 60)
print("EXPORTACIÓ")
print("=" * 60)
for fname, data in out_files.items():
    data.to_csv(fname, index=False, encoding='utf-8-sig', sep=';', decimal=',')
    print(f"✓ {fname}  ({len(data)} files)")
print(f"\n📁 Directori: {os.getcwd()}")
print("ℹ️ Format CSV: separador ';' i decimal ',' (obre directament a Excel en català/castellà)")
print("\n✅ Anàlisi completada!")

CALCUL DISTANCIA A APLICAR AL BANC

In [ ]:
# ============================================================
# Resultados del banco de ensayo separados en TEST 1 y TEST 2
# Test 1: pedaleo, bajas frecuencias (hasta ~5,3 Hz), sobre todo eje Y
# Test 2: frecuencias altas (hasta ~23-24 Hz)
# Pega esto en una celda de tu Colab (después de cargar df).
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.signal import butter, sosfiltfilt, welch, find_peaks

# ---------------- PARÁMETROS (ajusta a tu caso) ----------------
EJES = ["acc_y[mg]", "acc_z[mg]"]
EJES_VERTICALES = ["acc_z[mg]"]      # ejes donde se suma el peso a la fuerza
UNIDAD_ACC = "mg"                    # "mg", "g" o "m/s2"

TIME_COL = "time[us]"
TIME_FACTOR = 1e-6                   # microsegundos -> segundos
LABEL_COL = "label"                  # columna de etiqueta (o None)
T_INI, T_FIN = None, None            # tramo en s desde el inicio (None = todo)

# Cada test = una banda de frecuencias (Hz). Si cada test está en una etiqueta
# distinta de df["label"], pon aquí su nombre en "label"; None = usa todo el df.
TESTS = {
    "Test 1 · pedaleo (bajas frecuencias)": dict(f_min=0.8, f_max=6.0, label=None),
    "Test 2 · altas frecuencias":           dict(f_min=6.0, f_max=25.0, label=None),
}

MARGEN = 1.25            # margen de seguridad sobre el recorrido
MASA_KG = 4.5            # masa móvil total (motor + placa + tornillería)
PASO_HUSILLO_MM = 8.0    # mm por vuelta del husillo
G = 9.80665


# ---------------- FUNCIONES ----------------
def a_ms2(col):
    col = np.asarray(col, float)
    if UNIDAD_ACC == "mg":
        return col * 1e-3 * G
    if UNIDAD_ACC == "g":
        return col * G
    return col


def preparar(df, label):
    d = df
    if LABEL_COL is not None and label is not None:
        d = d[d[LABEL_COL] == label]
    t = d[TIME_COL].to_numpy(float) * TIME_FACTOR
    t = t - t[0]
    mask = np.ones(len(t), dtype=bool)
    if T_INI is not None:
        mask &= t >= T_INI
    if T_FIN is not None:
        mask &= t <= T_FIN
    d, t = d[mask], t[mask]
    fs = 1.0 / np.median(np.diff(t))
    return t, d, fs


def integrar(a_si, fs, f_min, f_max):
    """Aceleración filtrada, velocidad y desplazamiento (SI) en la banda dada."""
    a = np.asarray(a_si, float)
    a = a - a.mean()
    nyq = fs / 2
    sos = butter(4, [f_min / nyq, f_max / nyq], btype="band", output="sos")
    a = sosfiltfilt(sos, a)
    n = len(a)
    A = np.fft.rfft(a)
    f = np.fft.rfftfreq(n, 1 / fs)
    banda = f >= max(0.5 * f_min, 0.05)
    w = 2 * np.pi * np.where(f > 0, f, 1.0)
    v = np.fft.irfft(np.where(banda, A / (1j * w), 0), n)    # v = a/(jw)
    x = np.fft.irfft(np.where(banda, -A / w**2, 0), n)       # x = -a/w^2
    return a, v, x


def picos(a_f, fs, f_min, f_max, n=3):
    """Las n frecuencias con más energía dentro de la banda."""
    f, p = welch(a_f, fs, nperseg=min(len(a_f), int(fs * 20)))
    m = (f >= f_min) & (f <= f_max)
    idx, _ = find_peaks(p[m])
    if len(idx) == 0:
        return "-"
    orden = idx[np.argsort(p[m][idx])[::-1][:n]]
    return ", ".join(f"{f[m][i]:.1f}" for i in sorted(orden)) + " Hz"


def analizar(df, cfg):
    t, d, fs = preparar(df, cfg["label"])
    if len(t) < 100:
        raise ValueError("Muy pocas muestras: revisa label / T_INI / T_FIN")
    t_u = np.arange(0, t[-1], 1 / fs)          # malla de tiempo uniforme
    series, crudo, filas = {}, {}, []
    for eje in EJES:
        col = pd.Series(d[eje].to_numpy(float)).interpolate().bfill().ffill().to_numpy()
        a = np.interp(t_u, t, a_ms2(col))
        crudo[eje] = a - a.mean()
        series[eje] = integrar(a, fs, cfg["f_min"], cfg["f_max"])

    rms2 = {e: np.mean(series[e][0] ** 2) for e in EJES}
    total = sum(rms2.values())
    for eje in EJES:
        a_f, v, x = series[eje]
        x_mm, v_mms = x * 1e3, v * 1e3
        p2p = x_mm.max() - x_mm.min()
        a_pico = np.abs(a_f).max()
        v_max = np.abs(v_mms).max()
        f_din = MASA_KG * a_pico
        f_max = MASA_KG * (G + a_pico) if eje in EJES_VERTICALES else f_din
        filas.append({
            "Eje": eje,
            "% energía (vs otros ejes)": 100 * rms2[eje] / total,
            "Frecuencias dominantes": picos(a_f, fs, cfg["f_min"], cfg["f_max"]),
            "Acel. pico [g]": a_pico / G,
            "Acel. RMS [g]": np.sqrt(rms2[eje]) / G,
            "Recorrido pico-pico [mm]": p2p,
            "Recorrido robusto 99% [mm]": np.percentile(x_mm, 99.5) - np.percentile(x_mm, 0.5),
            "Carrera recomendada [mm]": p2p * MARGEN,
            "Vel. máx [mm/s]": v_max,
            "Vel. RMS [mm/s]": np.sqrt(np.mean(v_mms ** 2)),
            "Husillo máx [rpm]": v_max * 60 / PASO_HUSILLO_MM,
            "Fuerza dinámica [N]": f_din,
            "Fuerza máx (con peso en Z) [N]": f_max,
        })
    tabla = pd.DataFrame(filas).set_index("Eje")
    num = tabla.select_dtypes("number").columns
    tabla[num] = tabla[num].round(3)
    return tabla, t_u, series, crudo, fs, len(t)


# ---------------- EJECUCIÓN ----------------
resultados = {}
for nombre, cfg in TESTS.items():
    tabla, t_u, series, crudo, fs, n = analizar(df, cfg)
    resultados[nombre] = (t_u, series, crudo, fs, cfg)
    print("=" * 70)
    print(f"{nombre}  |  banda {cfg['f_min']}-{cfg['f_max']} Hz  |  fs = {fs:.1f} Hz  |  "
          f"{n} muestras  |  {t_u[-1]:.0f} s")
    print("=" * 70)
    display(tabla.T)

# ---------------- GRÁFICAS: TIEMPO ----------------
for nombre, (t_u, series, crudo, fs, cfg) in resultados.items():
    fig, ax = plt.subplots(len(EJES), 3, figsize=(15, 3.0 * len(EJES)), squeeze=False)
    fig.suptitle(nombre, fontsize=13)
    for i, eje in enumerate(EJES):
        a_f, v, x = series[eje]
        for j, (y, tit, col) in enumerate([(a_f / G, "aceleración [g]", "tab:green"),
                                           (v * 1e3, "velocidad [mm/s]", "tab:orange"),
                                           (x * 1e3, "desplazamiento [mm]", "tab:blue")]):
            ax[i, j].plot(t_u, y, lw=0.7, color=col)
            ax[i, j].set_title(f"{eje}: {tit}", fontsize=10)
            ax[i, j].set_xlabel("t [s]")
            ax[i, j].grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

# ---------------- GRÁFICA: ESPECTRO Y BANDAS ----------------
fig, ax = plt.subplots(1, len(resultados), figsize=(7 * len(resultados), 3.6), squeeze=False)
for k, (nombre, (t_u, series, crudo, fs, cfg)) in enumerate(resultados.items()):
    for eje in EJES:
        f, p = welch(crudo[eje] / G, fs, nperseg=min(len(t_u), int(fs * 20)))
        ax[0, k].semilogy(f, p, lw=0.9, label=eje)
    ax[0, k].axvspan(cfg["f_min"], cfg["f_max"], color="gray", alpha=0.15)
    ax[0, k].set_xlim(0, 40)
    ax[0, k].set_title(f"{nombre}: PSD aceleración [g²/Hz] (banda sombreada)", fontsize=10)
    ax[0, k].set_xlabel("Frecuencia [Hz]")
    ax[0, k].grid(alpha=0.3)
    ax[0, k].legend()
plt.tight_layout()
plt.show()

# ---------------- PERFILES PARA EL BANCO (CSV) ----------------
for nombre, (t_u, series, crudo, fs, cfg) in resultados.items():
    etiqueta = "test1" if nombre.startswith("Test 1") else "test2"
    perfil = pd.DataFrame({"t_s": t_u})
    for eje in EJES:
        a_f, v, x = series[eje]
        perfil[f"x_{eje}_mm"] = x * 1e3
        perfil[f"a_{eje}_g"] = a_f / G
    nombre_csv = f"perfil_{etiqueta}.csv"
    perfil.to_csv(nombre_csv, index=False)
    print(f"Guardado: {nombre_csv}  (posición en mm y aceleración en g, por eje)")